# S1 v2 pointer A: no option index + exact-rev JSD/logit MSE + curriculum 4-64 + temp floor (1x T4)

Gate vs s1_a v1: flips < 0.01, sel >= 0.875 / zs >= 0.933 (n=120). Saves ckpt_A.
1000 steps, 4000 rows, batch 4, Qwen3-0.6B-Base. Two forwards per step (symmetric order loss). About 90-100 min.

In [ ]:
import os, subprocess, sys, time, pathlib, json
REPO = "/kaggle/working/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "upp-p0", "https://github.com/nisaral/Soro-Boro.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "upp-p0"], check=False)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout, flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "huggingface_hub"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

In [ ]:
import shutil, torch
t0 = time.time()
cmd = [
    sys.executable, "veyra/training/p0_upp_ablation.py",
    "--s1", "--configs", "A", "--save-ckpt",
    "--base-model", "Qwen/Qwen3-0.6B-Base",
    "--train-rows", "4000", "--eval-rows", "120", "--batch", "4",
    "--max-seq-len", "832", "--max-steps", "1000", "--log-every", "50",
    "--out", "runs/s1_a",
]
print(" ".join(cmd), flush=True)
rc = subprocess.run(cmd).returncode
print(f"done in {(time.time()-t0)/60:.1f} min rc={rc}", flush=True)
if rc != 0:
    raise SystemExit(rc)
p = pathlib.Path("runs/s1_a/results.json")
print(p.read_text() if p.exists() else "no results")
ckpt = pathlib.Path("runs/s1_a/ckpt_A/head.pt")
if ckpt.exists():
    hs = torch.load(ckpt, map_location="cpu", weights_only=False)["head"]
    if "members.0.log_temp" in hs:
        print("final temp:", round(float(torch.exp(hs["members.0.log_temp"])), 5), flush=True)
shutil.make_archive("/kaggle/working/s1_a_results", "zip", "runs/s1_a")
print("wrote /kaggle/working/s1_a_results.zip", flush=True)